## Ejercicio 2: Escalamiento de tickets de soporte técnico
 
El agente trabaja en la mesa de ayuda y decide a qué nivel de soporte mandar cada ticket que
entra.
 
### Ficha PEAS
 
| | |
|---|---|
| **Percepción (S)** | Los minutos que el ticket lleva esperando, la urgencia que reportó el cliente (baja, media o alta) y si el cliente es premium o no. |
| **Acciones (A)** | Atender en nivel 1, escalar a nivel 2 o escalar a nivel 3. Si el dato llega raro, deriva el ticket a una persona. |
| **Entorno (E)** | La mesa de ayuda: la cola de tickets, los tres niveles de soporte con distinta capacidad y los acuerdos de tiempo de respuesta que el área tiene con los clientes. |
| **Objetivo** | Que cada ticket llegue al nivel que lo puede resolver a tiempo, sin usar de más a los técnicos especializados. |
| **Medida de desempeño (P)** | Cuántos tickets se resuelven dentro del tiempo prometido, cuánto demora en promedio la resolución, cuántos tickets se escalaron sin necesidad y qué tan conforme queda el cliente. |
 
**Tipo de agente:** reactivo simple, otra vez con reglas si-entonces. Ve un ticket a la vez y no toma
en cuenta cuántos tickets hay en la cola ni cómo le fue con los anteriores.
 
**¿Es racional?** Lo es mientras la urgencia que marca el cliente sea real. Si un cliente se
acostumbra a poner siempre "alta", el agente le va a escalar todo al nivel 3 y va a terminar saturando
al equipo más caro con problemas que el nivel 1 podía resolver.
 

 
### Justificación de las reglas
 
Pensando en el cliente, un problema de urgencia alta significa que algo se le cayó y no puede
trabajar, así que no tiene sentido hacerlo esperar en cola: eso va directo al nivel 3. Pensando en el
equipo de soporte, escalar todo sería un desastre, porque los niveles 2 y 3 son pocos y caros, así que
lo de urgencia baja y media arranca en el nivel 1 y solo sube si ya esperó demasiado. El cliente
premium no tiene reglas diferentes, lo que hace es que los tiempos se le cuenten a la mitad, que es
justamente lo que está pagando con su plan.
 
**Reglas:**
 
| Urgencia | Tiempo de espera (cliente normal) | Tiempo de espera (cliente premium) | Decisión |
|---|---|---|---|
| alta | cualquiera | cualquiera | escalar a nivel 3 |
| media | 60 min o más | 30 min o más | escalar a nivel 3 |
| media | 20 min o más | 10 min o más | escalar a nivel 2 |
| media | menos de eso | menos de eso | atender en nivel 1 |
| baja | 120 min o más | 60 min o más | escalar a nivel 2 |
| baja | menos de eso | menos de eso | atender en nivel 1 |
 

 
### Código
 

In [1]:
def agente_soporte(tiempo_espera_minutos, nivel_urgencia, cliente_premium):
    urgencia = nivel_urgencia.lower()
 
    # si la urgencia no es una de las tres esperadas, mejor que lo vea una persona
    if urgencia not in ["baja", "media", "alta"]:
        return "derivar a una persona", "el nivel de urgencia '" + nivel_urgencia + "' no esta contemplado"
 
    # el cliente premium tiene la mitad de tolerancia de espera
    if cliente_premium:
        tope_nivel3 = 30
        tope_nivel2 = 10
        tope_baja = 60
        tipo = "cliente premium"
    else:
        tope_nivel3 = 60
        tope_nivel2 = 20
        tope_baja = 120
        tipo = "cliente normal"
 
    t = str(tiempo_espera_minutos)
 
    if urgencia == "alta":
        return "escalar a nivel 3", "urgencia alta reportada por el cliente, se escala de inmediato"
 
    if urgencia == "media":
        if tiempo_espera_minutos >= tope_nivel3:
            return "escalar a nivel 3", "urgencia media y ya lleva " + t + " min esperando (" + tipo + ")"
        elif tiempo_espera_minutos >= tope_nivel2:
            return "escalar a nivel 2", "urgencia media con " + t + " min de espera (" + tipo + ")"
        else:
            return "atender en nivel 1", "urgencia media pero recien lleva " + t + " min, el nivel 1 puede verlo"
 
    # urgencia baja
    if tiempo_espera_minutos >= tope_baja:
        return "escalar a nivel 2", "urgencia baja pero lleva " + t + " min en cola (" + tipo + "), ya es demasiado"
    else:
        return "atender en nivel 1", "urgencia baja y solo " + t + " min de espera, lo resuelve el nivel 1"
        